In [50]:
import numpy as np
import pandas as pd

In [51]:
data = pd.read_csv('train.csv')
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [161]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0].astype(int)
X_dev = data_dev[1:n]

data_train = data[1000:m].T
Y_train = data_train[0].astype(int)
X_train = data_train[1:n]

X_train = X_train / 255.0
X_dev = X_dev / 255.0


print(x_train.shape)
print(y_train.shape)
print(np.unique(y_train, return_counts=True))


w1, b1, w2, b2 = gd(X_train, Y_train, 2000, 0.1)

(784, 41000)
(41000,)
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([4032, 4583, 4080, 4235, 3988, 3689, 4055, 4300, 3971, 4067]))
iteration: 0
Accuracy: 0.12170731707317073
iteration: 50
Accuracy: 0.6338292682926829
iteration: 100
Accuracy: 0.790609756097561
iteration: 150
Accuracy: 0.8472926829268292
iteration: 200
Accuracy: 0.8711219512195122
iteration: 250
Accuracy: 0.8827073170731707
iteration: 300
Accuracy: 0.8906585365853659
iteration: 350
Accuracy: 0.8959268292682927
iteration: 400
Accuracy: 0.8996341463414634
iteration: 450
Accuracy: 0.9028780487804878
iteration: 500
Accuracy: 0.9054634146341464
iteration: 550
Accuracy: 0.9080487804878049
iteration: 600
Accuracy: 0.9107560975609756
iteration: 650
Accuracy: 0.9132439024390244
iteration: 700
Accuracy: 0.9148780487804878
iteration: 750
Accuracy: 0.9165121951219513
iteration: 800
Accuracy: 0.9175609756097561
iteration: 850
Accuracy: 0.9192926829268293
iteration: 900
Accuracy: 0.9206829268292683
iteration: 950
Accuracy: 0.921975609

In [160]:
def init_params():
    w1 = np.random.randn(128, 784) * 0.01
    b1 = np.zeros((128, 1))
    w2 = np.random.randn(10, 128) * 0.01
    b2 = np.zeros((10, 1))
    return w1, b1, w2, b2
def ReLU(z):
    return np.maximum(0, z)

def Softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

def derive_ReLU(z):
    return z > 0

def forward_pass(w1, w2, b1, b2, X):
    z1 = w1.dot(X) + b1
    A1 = ReLU(z1)
    z2 = w2.dot(A1) + b2
    A2 = Softmax(z2)
    return z1, A1, z2, A2

def one_hot(y):
    one_hot_y = np.zeros((y.size, y.max() + 1))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y.T

def back_prop(z1, A1, z2, A2, w2, y, X):
    m = y.size
    Y = one_hot(y)

    dz2 = A2 - Y
    dw2 = dz2.dot(A1.T) / m
    db2 = np.sum(dz2, axis=1, keepdims=True) / m

    dz1 = w2.T.dot(dz2) * derive_ReLU(z1)
    dw1 = dz1.dot(X.T) / m
    db1 = np.sum(dz1, axis=1, keepdims=True) / m

    return dw1, db1, dw2, db2

def update_params(w1, b1, w2, b2, dw1, db1, dw2, db2, alpha):
    w1 = w1 - alpha * dw1
    b1 = b1 - alpha * db1
    w2 = w2 - alpha * dw2
    b2 = b2 - alpha * db2
    return w1, b1, w2, b2

def Predictions(A2):
    return np.argmax(A2, axis=0)

def Accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def gd(X, Y, iterations, alpha):
    w1, b1, w2, b2 = init_params()

    for i in range(iterations):
        z1, A1, z2, A2 = forward_pass(w1, w2, b1, b2, X)

        dw1, db1, dw2, db2 = back_prop(
            z1, A1, z2, A2, w2, Y, X
        )

        w1, b1, w2, b2 = update_params(
            w1, b1, w2, b2,
            dw1, db1, dw2, db2,
            alpha
        )

        if i % 50 == 0:
            predictions = Predictions(A2)
            print("iteration:", i)
            print("Accuracy:", Accuracy(predictions, Y))

    return w1, b1, w2, b2

In [166]:
np.savez(
    "mnist_model.npz",
    w1=w1,
    b1=b1,
    w2=w2,
    b2=b2
)